<a href="https://colab.research.google.com/github/putriikrima2/Dhaka-AQI-Forecasting/blob/main/Revision_Analysis_AQI_Dhaka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Incremental Value of Meteorological Predictors for Next-Day AQI Forecasting in Dhaka

This notebook reproduces **every** table and figure of the revised manuscript:

| Section | Content | In the manuscript |
|---|---|---|
| 0–2 | Setup, data, feature engineering, chronological split | Sections 2.1–2.2 |
| 3 | Grid tuning, 2025 test, baselines (persistence, seasonal-naïve, climatology), ARX | Tables 3–4 |
| 4 | Diebold–Mariano test (HLN) and block bootstrap | Table 5 |
| 5–7 | 10 seeds, fixed-configuration ablation, rolling origin | Table 6 |
| 8 | Seasonal breakdown and bias | Table 7 |
| 9 | Meteorological-lag sensitivity and target-day weather | Table 6 |
| 10 | Lag selection on an inner split | Table 2 |
| 11 | Grouped permutation importance | Figure 5 |
| 12 | Data-source diagnostics (PM2.5/PM10 ratio, CAMS 49R1) | Section 3.1 |
| 13 | Pollutant-lag sensitivity (candidate C) | Table 6 |
| 14 | Summary of numbers in manuscript order | – |
| 15 | Figures 1–7 (300 dpi) | Figures 1–7 |

Versions used in the manuscript: Python 3.13.18, pandas 2.2.3, NumPy 2.4.6, scikit-learn 1.6.1, XGBoost 3.4.1, LightGBM 4.6.0 (Google Colab).
Total run time is about 6–10 minutes. Run with **Runtime → Run all**.

In [1]:
import pandas as pd

CSV = "https://raw.githubusercontent.com/putriikrima2/Dhaka-AQI-Forecasting/refs/heads/main/data/Dhaka_Climate_AQI_Data.csv"

df_check = pd.read_csv(CSV)

print("Dataset berhasil dibaca:", df_check.shape)
print("Kolom:", df_check.columns.tolist())


Dataset berhasil dibaca: (1096, 10)
Kolom: ['Date', 'District', 'Latitude', 'Longitude', 'Avg_Temperature_C', 'Rainfall_mm', 'Max_Wind_Speed_kmh', 'PM10', 'PM2.5', 'AQI_Level']


## 1 Libraries and software versions
Versions are printed for reporting in Section 2.7 of the manuscript.

In [ ]:
import json, sys, platform, warnings, itertools
import numpy as np, pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")

BACKEND={}
try:
    from xgboost import XGBRegressor; import xgboost; BACKEND["xgb"]="xgboost "+xgboost.__version__
except ImportError:
    raise ImportError("Install xgboost: !pip install xgboost")
try:
    from lightgbm import LGBMRegressor; import lightgbm; BACKEND["lgb"]="lightgbm "+lightgbm.__version__
except ImportError:
    raise ImportError("Install lightgbm: !pip install lightgbm")
import sklearn; BACKEND["sklearn"]=sklearn.__version__; BACKEND["python"]=platform.python_version()
BACKEND["pandas"]=pd.__version__; BACKEND["numpy"]=np.__version__
print(BACKEND)

## 2 Data, feature engineering (lags 0–2), next-day AQI target, and chronological split
Train: target ≤ 30 Jun 2024 (544) · Validation: Jul–Dec 2024 (184) · Test: 2025 (365). S1 = AQI, PM2.5, PM10 (9 features); S2 = S1 + temperature, rainfall, wind (18 features).

In [ ]:
# ---------------- data ----------------
df=pd.read_csv(CSV); df["Date"]=pd.to_datetime(df["Date"])
data=df.set_index("Date").sort_index().asfreq("D")
COLS={"AQI":"AQI_Level","PM25":"PM2.5","PM10":"PM10","Temp":"Avg_Temperature_C","Rain":"Rainfall_mm","Wind":"Max_Wind_Speed_kmh"}
POL=("AQI","PM25","PM10"); METV=("Temp","Rain","Wind")
def build(pl=(0,1,2), ml=(0,1,2), lead_met=False):
    fe=pd.DataFrame(index=data.index)
    for n,c in COLS.items():
        for k in (pl if n in POL else ml): fe[f"{n}_lag{k}"]=data[c].shift(k)
    if lead_met:
        for n in METV: fe[f"{n}_lead1"]=data[COLS[n]].shift(-1)
    fe["y"]=data["AQI_Level"].shift(-1)
    return fe
def masks(idx):
    tt=idx+pd.Timedelta(days=1)
    tr=tt<=pd.Timestamp("2024-06-30"); va=(tt>=pd.Timestamp("2024-07-01"))&(tt<=pd.Timestamp("2024-12-31")); te=tt>=pd.Timestamp("2025-01-01")
    return np.asarray(tr),np.asarray(va),np.asarray(te)
S1=[f"{v}_lag{k}" for v in POL for k in (0,1,2)]
S2=S1+[f"{v}_lag{k}" for v in METV for k in (0,1,2)]
fe=build().dropna(); tr,va,te=masks(fe.index); trva=tr|va
y=fe["y"].values; tdate=fe.index+pd.Timedelta(days=1)

GRID={"RF":{"n_estimators":[150,300],"max_depth":[10,15],"min_samples_leaf":[2,4]},
      "XGB":{"n_estimators":[100,300,500],"max_depth":[3,5],"learning_rate":[0.01,0.05,0.1]},
      "LGB":{"n_estimators":[100,300,500],"num_leaves":[15,31],"learning_rate":[0.01,0.05,0.1]}}
def make(alg,p,seed=42):
    if alg=="RF": return RandomForestRegressor(**p,random_state=seed,n_jobs=-1)
    if alg=="XGB":
        if "EMU" in BACKEND["xgb"]: return XGBEmu(**p)
        return XGBRegressor(**p,objective="reg:squarederror",random_state=seed,n_jobs=-1,verbosity=0)
    if alg=="LGB":
        if "EMU" in BACKEND["lgb"]:
            return HistGradientBoostingRegressor(max_iter=p["n_estimators"],max_leaf_nodes=p["num_leaves"],learning_rate=p["learning_rate"],
                   min_samples_leaf=p.get("min_child_samples",10),l2_regularization=0,max_bins=255,early_stopping=False,random_state=seed)
        q=dict(p); q.setdefault("min_child_samples",10)
        return LGBMRegressor(**q,max_depth=-1,random_state=seed,n_jobs=-1,verbosity=-1)
    if alg=="ARX": return LinearRegression()
def rmse(a,b): return float(np.sqrt(mean_squared_error(a,b)))
def metrics(yt,yp): return dict(MAE=float(mean_absolute_error(yt,yp)),RMSE=rmse(yt,yp),R2=float(r2_score(yt,yp)),Bias=float(np.mean(yt-yp)))
def combos(alg): g=GRID[alg]; return [dict(zip(g,v)) for v in itertools.product(*g.values())]
def fitpred(alg,p,feats,trm,tem,seed=42):
    X=fe[feats].values; m=make(alg,p,seed).fit(X[trm],y[trm]); return m.predict(X[tem]),m
R={"backend":BACKEND}

## 3 Grid tuning, 2025 test evaluation, baselines, and ARX
The best configuration is chosen by validation RMSE (MAE as tie-breaker), refitted on train + validation, and tested once on 2025. → Tables 3 and 4.

In [ ]:
# ---------- 1. tuning + final test ----------
tun={};best={};preds={}
for alg in ("RF","XGB","LGB"):
    for sc,F in (("S1",S1),("S2",S2)):
        rows=[]
        for p in combos(alg):
            pv,_=fitpred(alg,p,F,tr,va); rows.append((rmse(y[va],pv),float(mean_absolute_error(y[va],pv)),p))
        rows.sort(key=lambda r:(r[0],r[1])); tun[(alg,sc)]=rows; best[(alg,sc)]=rows[0][2]
        pt,_=fitpred(alg,rows[0][2],F,trva,te); preds[(alg,sc)]=pt
for sc,F in (("S1",S1),("S2",S2)):
    pt,_=fitpred("ARX",{},F,trva,te); preds[("ARX",sc)]=pt
yt=y[te]
preds[("Persistence","-")]=fe["AQI_lag0"].values[te]
sn=data["AQI_Level"].shift(6).reindex(fe.index).values  # same weekday of previous week for target t+1
preds[("SeasonalNaive7","-")]=sn[te]
clim=pd.Series(y[trva],index=tdate[trva]).groupby(lambda d:d.month).mean()
preds[("Climatology","-")]=np.array([clim[d.month] for d in tdate[te]])
R["validation"]={f"{a}-{s}":dict(RMSE=r[0][0],MAE=r[0][1],params=r[0][2]) for (a,s),r in tun.items()}
R["test"]={f"{a}-{s}":metrics(yt,p) for (a,s),p in preds.items()}
print(pd.DataFrame(R["test"]).T.round(3))

## 4 Paired inference: Diebold–Mariano (HLN correction) and moving-block bootstrap
→ Table 5.

In [ ]:
# ---------- 2. DM test + block bootstrap ----------
def dm_test(e1,e2,loss="sq",h=1,L=None):
    d=(e1**2-e2**2) if loss=="sq" else (np.abs(e1)-np.abs(e2)); n=len(d); L=int(np.floor(n**(1/3))) if L is None else L
    dc=d-d.mean(); g0=np.dot(dc,dc)/n; s=g0
    for k in range(1,L+1): s+=2*(1-k/(L+1))*np.dot(dc[k:],dc[:-k])/n
    dm=d.mean()/np.sqrt(s/n); hln=dm*np.sqrt((n+1-2*h+h*(h-1)/n)/n)
    p=2*stats.t.sf(abs(hln),df=n-1); return float(hln),float(p)
def block_boot(e1,e2,B=5000,bl=14,seed=2025):
    rng=np.random.RandomState(seed); n=len(e1); nb=int(np.ceil(n/bl)); out=[];outp=[]
    for _ in range(B):
        st=rng.randint(0,n-bl+1,nb); idx=np.concatenate([np.arange(s,s+bl) for s in st])[:n]
        r1=np.sqrt(np.mean(e1[idx]**2)); r2=np.sqrt(np.mean(e2[idx]**2)); out.append(r1-r2); outp.append((r1-r2)/r1*100)
    return [float(np.percentile(out,2.5)),float(np.percentile(out,97.5))],[float(np.percentile(outp,2.5)),float(np.percentile(outp,97.5))],float(np.mean(np.array(out)>0))
R["paired"]={}
for alg in ("RF","XGB","LGB","ARX"):
    e1=yt-preds[(alg,"S1")]; e2=yt-preds[(alg,"S2")]
    dms,pms=dm_test(e1,e2,"sq"); dma,pma=dm_test(e1,e2,"abs"); ci,cip,pp=block_boot(e1,e2)
    dd=np.abs(e1)-np.abs(e2)
    R["paired"][alg]=dict(dRMSE=rmse(yt,preds[(alg,"S1")])-rmse(yt,preds[(alg,"S2")]),
        dRMSEpct=(rmse(yt,preds[(alg,"S1")])-rmse(yt,preds[(alg,"S2")]))/rmse(yt,preds[(alg,"S1")])*100,
        DM_sq=dms,p_sq=pms,DM_abs=dma,p_abs=pma,CI_dRMSE=ci,CI_pct=cip,boot_share_pos=pp,
        days_S2_better=int((dd>0).sum()),mean_d=float(dd.mean()))
print(pd.DataFrame(R["paired"]).T)

## 5 Stability across 10 random seeds
XGBoost and LightGBM without subsampling are deterministic (SD = 0). → Table 6, row 1.

In [ ]:
# ---------- 3. seeds ----------
R["seeds"]={}
for alg in ("RF","XGB","LGB"):
    for sc,F in (("S1",S1),("S2",S2)):
        rs=[rmse(yt,fitpred(alg,best[(alg,sc)],F,trva,te,seed=s)[0]) for s in range(10)]
        R["seeds"][f"{alg}-{sc}"]=dict(mean=float(np.mean(rs)),sd=float(np.std(rs,ddof=1)),min=float(min(rs)),max=float(max(rs)))
    a=np.array([rmse(yt,fitpred(alg,best[(alg,"S1")],S1,trva,te,seed=s)[0])-rmse(yt,fitpred(alg,best[(alg,"S2")],S2,trva,te,seed=s)[0]) for s in range(10)])
    R["seeds"][f"{alg}-dRMSE"]=dict(mean=float(a.mean()),sd=float(a.std(ddof=1)),n_pos=int((a>0).sum()))
print(R["seeds"])

## 6 Fixed-configuration ablation
Every grid configuration is used for both S1 and S2, so the feature effect is separated from the tuning effect. → Table 6, rows 2–3.

In [ ]:
# ---------- 4. fixed-configuration ablation ----------
R["fixed"]={}
for alg in ("RF","XGB","LGB"):
    rows=[]
    for p in combos(alg):
        v1=rmse(y[va],fitpred(alg,p,S1,tr,va)[0]); v2=rmse(y[va],fitpred(alg,p,S2,tr,va)[0])
        t1=rmse(yt,fitpred(alg,p,S1,trva,te)[0]); t2=rmse(yt,fitpred(alg,p,S2,trva,te)[0])
        rows.append(dict(p=p,v1=v1,v2=v2,t1=t1,t2=t2,dval=(v1-v2)/v1*100,dtest=(t1-t2)/t1*100))
    D=pd.DataFrame(rows); D["vm"]=(D.v1+D.v2)/2; top=D.nsmallest(5,"vm")
    sameS1=D[D.p.apply(lambda q:q==best[(alg,"S1")])].iloc[0]; sameS2=D[D.p.apply(lambda q:q==best[(alg,"S2")])].iloc[0]
    R["fixed"][alg]=dict(n=len(D),val_median=float(D.dval.median()),val_share_pos=float((D.dval>0).mean()),
        test_median=float(D.dtest.median()),test_q1=float(D.dtest.quantile(.25)),test_q3=float(D.dtest.quantile(.75)),test_share_pos=float((D.dtest>0).mean()),
        top5_val=float(top.dval.mean()),top5_test=float(top.dtest.mean()),cfgS1_test=float(sameS1.dtest),cfgS2_test=float(sameS2.dtest),
        common_cfg=top.iloc[0].p)
print(pd.DataFrame(R["fixed"]).T)

## 7 Rolling-origin evaluation (18 monthly origins, Jul 2024–Dec 2025)
→ Table 6, row 4.

In [ ]:
# ---------- 5. rolling-origin (monthly, expanding window, common config) ----------
R["rolling"]={}
months=pd.period_range("2024-07","2025-12",freq="M")
for alg in ("RF","XGB","LGB","ARX"):
    p=R["fixed"][alg]["common_cfg"] if alg!="ARX" else {}
    E1=[];E2=[];per=[]
    for m in months:
        tm=np.asarray(tdate.to_period("M")==m); trm=np.asarray(tdate<m.start_time)
        e1=y[tm]-fitpred(alg,p,S1,trm,tm)[0]; e2=y[tm]-fitpred(alg,p,S2,trm,tm)[0]
        E1.append(e1);E2.append(e2); r1=np.sqrt(np.mean(e1**2)); r2=np.sqrt(np.mean(e2**2)); per.append((str(m),float(r1),float(r2),float((r1-r2)/r1*100)))
    E1=np.concatenate(E1);E2=np.concatenate(E2); dms,pms=dm_test(E1,E2)
    r1=np.sqrt(np.mean(E1**2)); r2=np.sqrt(np.mean(E2**2))
    R["rolling"][alg]=dict(per_month=per,RMSE_S1=float(r1),RMSE_S2=float(r2),dpct=float((r1-r2)/r1*100),
        months_S2_better=int(sum(1 for q in per if q[3]>0)),n_months=len(per),DM=dms,p=pms,
        val_part=float(np.mean([q[3] for q in per[:6]])),test_part=float(np.mean([q[3] for q in per[6:]])))
print({k:(round(v["dpct"],2),v["months_S2_better"],round(v["p"],3)) for k,v in R["rolling"].items()})

## 8 Seasonal breakdown and bias
→ Table 7.

In [ ]:
# ---------- 6. seasonal breakdown & bias (test 2025) ----------
seas={"DJF (winter)":[12,1,2],"MAM (pre-monsoon)":[3,4,5],"JJAS (monsoon)":[6,7,8,9],"ON (post-monsoon)":[10,11]}
tm=np.array([d.month for d in tdate[te]]); R["season"]={}
for alg in ("RF","XGB","LGB","ARX"):
    R["season"][alg]={}
    for s,ms in seas.items():
        k=np.isin(tm,ms); r1=rmse(yt[k],preds[(alg,"S1")][k]); r2=rmse(yt[k],preds[(alg,"S2")][k])
        R["season"][alg][s]=dict(n=int(k.sum()),RMSE_S1=r1,RMSE_S2=r2,dpct=(r1-r2)/r1*100,
           bias_S1=float(np.mean(yt[k]-preds[(alg,"S1")][k])),bias_S2=float(np.mean(yt[k]-preds[(alg,"S2")][k])))
pr=preds[("Persistence","-")]
R["season"]["Persistence"]={s:dict(RMSE=rmse(yt[np.isin(tm,ms)],pr[np.isin(tm,ms)])) for s,ms in seas.items()}

## 9 Meteorological-lag sensitivity and target-day weather (NWP proxy)
→ Table 6, rows 6–7.

In [ ]:
# ---------- 7. meteorological lag sensitivity + target-day meteorology (oracle NWP proxy) ----------
R["metlag"]={}
variants={"none (S1)":None,"{0}":(0,),"{0,1}":(0,1),"{0,1,2} (S2)":(0,1,2),"{0,...,6}":tuple(range(7)),"{0,1,2}+t+1":"lead"}
for alg in ("RF","XGB","LGB","ARX"):
    p=R["fixed"][alg]["common_cfg"] if alg!="ARX" else {}
    R["metlag"][alg]={}
    for nm,ml in variants.items():
        if ml is None: f2=build(); F=S1
        elif ml=="lead": f2=build(lead_met=True); F=S2+[f"{v}_lead1" for v in METV]
        else: f2=build(ml=ml); F=S1+[f"{v}_lag{k}" for v in METV for k in ml]
        f2=f2.dropna(); f2=f2.loc[f2.index>=pd.Timestamp("2023-01-07")]  # common sample start for all variants (max lag 6)
        a,b,c=masks(f2.index); X=f2[F].values; yy=f2.y.values
        v=rmse(yy[b],make(alg,p).fit(X[a],yy[a]).predict(X[b])); t=rmse(yy[c],make(alg,p).fit(X[a|b],yy[a|b]).predict(X[c]))
        R["metlag"][alg][nm]=dict(val=v,test=t)
print(pd.DataFrame({a:{k:round(v["test"],3) for k,v in d.items()} for a,d in R["metlag"].items()}))

## 10 Pollutant-lag selection on an inner split
The Jul–Dec 2024 validation period is not used twice; also re-tested with `min_child_samples` = 10. → Table 2.

In [ ]:
# ---------- 8. lag selection with an inner split (validation set not reused) ----------
cand={"A":[0,1,2],"B":[0,1,2,6],"C":list(range(7)),"D":[0,1,2,10]}
R["lagsel"]={}
LAGCFG={"RF":dict(n_estimators=300,min_samples_leaf=2),"XGB":dict(n_estimators=300,max_depth=3,learning_rate=0.05),
        "LGB20":dict(n_estimators=300,max_depth=5,num_leaves=15,learning_rate=0.05,min_child_samples=20),
        "LGB10":dict(n_estimators=300,max_depth=5,num_leaves=15,learning_rate=0.05,min_child_samples=10)}
def mk_lag(alg,p):
    if alg=="RF": return RandomForestRegressor(**p,random_state=42,n_jobs=-1)
    if alg=="XGB": return make("XGB",p)
    if "EMU" in BACKEND["lgb"]:
        return HistGradientBoostingRegressor(max_iter=p["n_estimators"],max_leaf_nodes=p["num_leaves"],max_depth=p["max_depth"],learning_rate=p["learning_rate"],min_samples_leaf=p["min_child_samples"],l2_regularization=0,early_stopping=False,random_state=42)
    return LGBMRegressor(**p,random_state=42,n_jobs=-1,verbosity=-1)
base=None
for c,L in cand.items():
    f2=pd.DataFrame(index=data.index)
    for n in POL:
        for k in L: f2[f"{n}_lag{k}"]=data[COLS[n]].shift(k)
    f2["y"]=data["AQI_Level"].shift(-1); f2=f2.dropna(); f2=f2.loc[f2.index>=pd.Timestamp("2023-01-11")]
    td=f2.index+pd.Timedelta(days=1); itr=np.asarray(td<=pd.Timestamp("2023-12-31")); iva=np.asarray((td>=pd.Timestamp("2024-01-01"))&(td<=pd.Timestamp("2024-06-30")))
    otr=np.asarray(td<=pd.Timestamp("2024-06-30")); ova=np.asarray((td>=pd.Timestamp("2024-07-01"))&(td<=pd.Timestamp("2024-12-31")))
    X=f2.drop(columns="y").values; yy=f2.y.values; R["lagsel"][c]={}
    for alg,p in LAGCFG.items():
        a2=alg[:3]
        R["lagsel"][c][alg]=dict(inner=rmse(yy[iva],mk_lag(a2,p).fit(X[itr],yy[itr]).predict(X[iva])),
                                 outer=rmse(yy[ova],mk_lag(a2,p).fit(X[otr],yy[otr]).predict(X[ova])))
print(pd.DataFrame({c:{a:round(v["inner"],3) for a,v in d.items()} for c,d in R["lagsel"].items()}))

## 11 Grouped permutation importance (2024 validation and 2025 test)
All lags of one variable are permuted together (10 repeats). → Figure 5.

In [ ]:
# ---------- 9. grouped permutation importance ----------
def grouped_pi(model,X,yy,feats,groups,R_=10,seed=42):
    rng=np.random.RandomState(seed); ref=rmse(yy,model.predict(X)); out={}
    for g,cols in groups.items():
        ix=[feats.index(c) for c in cols]; vals=[]
        for r in range(R_):
            Xp=X.copy(); perm=rng.permutation(len(X)); Xp[:,ix]=X[perm][:,ix]; vals.append(rmse(yy,model.predict(Xp))-ref)
        out[g]=(float(np.mean(vals)),float(np.std(vals,ddof=1)))
    return out
groups={g:[f"{g}_lag{k}" for k in (0,1,2)] for g in POL+METV}; groups["Meteorology (all)"]=[f for f in S2 if f.split("_")[0] in METV]
R["gpi"]={}
X2=fe[S2].values
for alg in ("RF","XGB","LGB"):
    p=best[(alg,"S2")]
    mv=make(alg,p).fit(X2[tr],y[tr]); mt=make(alg,p).fit(X2[trva],y[trva])
    R["gpi"][alg]=dict(val=grouped_pi(mv,X2[va],y[va],S2,groups),test=grouped_pi(mt,X2[te],y[te],S2,groups))
print({a:{g:round(v[0],2) for g,v in d["test"].items()} for a,d in R["gpi"].items()})

## 12 Data-source diagnostics
PM2.5/PM10 ratio before and after the CAMS cycle 49R1 upgrade (12 Nov 2024). → Section 3.1.

In [ ]:
# ---------- 10. data provenance diagnostics ----------
ratio=data["PM2.5"]/data["PM10"]
R["ratio"]=dict(pre=float(ratio[:"2024-11-11"].mean()),pre_sd=float(ratio[:"2024-11-11"].std()),post=float(ratio["2024-11-13":].mean()),post_sd=float(ratio["2024-11-13":].std()))

## 13 Pollutant-lag sensitivity (candidate C, lags 0–6)
→ Table 6, row 5.

In [ ]:
# ---------- 11. pollutant-lag sensitivity (candidate C) ----------
R["polsens"]={}
for pl,lab in [((0,1,2),"A"),(tuple(range(7)),"C")]:
    f2=build(pl=pl,ml=(0,1,2)).dropna(); f2=f2.loc[f2.index>=pd.Timestamp("2023-01-07")]
    a,b,c=masks(f2.index); yy=f2.y.values
    Pf=[f"{v_}_lag{k}" for v_ in POL for k in pl]; Mf=[f"{v_}_lag{k}" for v_ in METV for k in (0,1,2)]
    for alg in ("RF","XGB","LGB","ARX"):
        p=R["fixed"][alg]["common_cfg"] if alg!="ARX" else {}
        r=[rmse(yy[c],make(alg,p).fit(f2[F].values[a|b],yy[a|b]).predict(f2[F].values[c])) for F in (Pf,Pf+Mf)]
        R["polsens"][f"{lab}-{alg}"]=dict(S1=r[0],S2=r[1],dpct=(r[0]-r[1])/r[0]*100)

## 14 Summary of numbers in manuscript order
This block prints every number used in Tables 4–7 and the text, then saves `results.json` and `test_predictions.csv`.

In [ ]:
# ---------- 12. summary in manuscript order ----------
def f2_(x): return f"{x:.2f}"
print("\n================ VALUES FOR THE MANUSCRIPT ================")
print("Software:",BACKEND)
print("Table 4 (test): model, MAE, RMSE, R2, Bias")
for k,m in R["test"].items(): print(" ",k,f2_(m["MAE"]),f2_(m["RMSE"]),f"{m['R2']:.3f}",f2_(m["Bias"]))
print("Table 5 (paired): dRMSE%, DM p sq/abs, CI%")
for a,m in R["paired"].items(): print(" ",a,f2_(m["dRMSEpct"]),f"{m['p_sq']:.3f}/{m['p_abs']:.3f}",[round(x,2) for x in m["CI_pct"]],m["days_S2_better"])
print("Table 6 (robustness)")
for a in ("RF","XGB","LGB"): print(" seeds",a,R["seeds"][f"{a}-dRMSE"])
for a,m in R["fixed"].items(): print(" fixed",a,f2_(m["test_median"]),f"{m['test_share_pos']*m['n']:.0f}/{m['n']}","val>0:",f"{m['val_share_pos']*m['n']:.0f}/{m['n']}")
for a,m in R["rolling"].items(): print(" rolling",a,f2_(m["dpct"]),m["months_S2_better"],f"{m['p']:.3f}")
for k,m in R["polsens"].items(): print(" polsens",k,f2_(m["dpct"]))
for a,d in R["metlag"].items():
    s1=d["none (S1)"]["test"]; print(" metlag",a,{k:f2_((s1-v_["test"])/s1*100) for k,v_ in d.items() if k!="none (S1)"})
print("Table 7 (season dRMSE%)")
for a,d in R["season"].items(): print(" ",a,{k:f2_(v_.get("dpct",v_.get("RMSE"))) for k,v_ in d.items()})
print("Lag screen (inner/outer)")
for c,d in R["lagsel"].items(): print(" ",c,{a:(f2_(v_["inner"]),f2_(v_["outer"])) for a,v_ in d.items()})
print("Grouped PI (mean, sd)")
for a,d in R["gpi"].items(): print(" ",a,{p:{g:(f2_(m[0]),f2_(m[1])) for g,m in d[p].items()} for p in d})

# keep predictions for figures
pd.DataFrame({f"{a}-{s}":p for (a,s),p in preds.items()}|{"y":yt},index=tdate[te]).to_csv("test_predictions.csv")
R["best"]={f"{a}-{s}":p for (a,s),p in best.items()}
json.dump(R,open("results.json","w"),indent=1,default=str)
print("Saved: results.json and test_predictions.csv")

## 15 Figures 1–7 (300 dpi)
Figures are saved in the `figures/` folder. File name → figure number in the manuscript: `fig1_workflow` → Figure 1, `fig2_timeseries` → Figure 2, `fig7_aqi_recon` → Figure 3, `fig3_metrics` → Figure 4, `fig4_gpi` → Figure 5, `fig5_predictions` → Figure 6, `fig6_residuals` → Figure 7.

In [ ]:
import os, json, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from types import SimpleNamespace
A=SimpleNamespace(csv=CSV, original_main=False)
os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","Liberation Serif","DejaVu Serif"],"font.size":8,"axes.titlesize":8,
 "axes.labelsize":8,"xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":7,"axes.spines.top":False,"axes.spines.right":False,
 "axes.linewidth":0.6,"xtick.major.width":0.6,"ytick.major.width":0.6,"savefig.dpi":300,"savefig.bbox":"tight","savefig.pad_inches":0.02})
C1,C2,GR,DK="#257de0","#f47230","#8a8a8a","#191919"
R=json.load(open("results.json")); P=pd.read_csv("test_predictions.csv",index_col=0,parse_dates=True)
df=pd.read_csv(A.csv,parse_dates=["Date"]).set_index("Date")
W=6.3
def lab(ax,s,x=-0.07,y=1.02): ax.text(x,y,s,transform=ax.transAxes,fontweight="bold",va="bottom",ha="right")

In [ ]:
# ---- Fig 1 workflow (clean flowchart; text auto-fitted inside boxes) ----
import matplotlib.pyplot as _plt
from matplotlib.patches import Rectangle as _Rect
def make_fig1(path="figures/fig1_workflow.png"):
    FW,FH=6.3,3.55
    fig=_plt.figure(figsize=(FW,FH)); ax=fig.add_axes([0,0,1,1]); ax.set_xlim(0,FW); ax.set_ylim(0,FH); ax.axis("off")
    SANS={"family":"sans-serif","fontname":"DejaVu Sans"}
    EC="#333333"; boxes=[]
    FS=6.3
    def box(x,y,w,h,title,body,fc="white"):
        ax.add_patch(_Rect((x,y),w,h,fc=fc,ec=EC,lw=0.7))
        nb=body.count("\n")+1; lh=FS*1.32/72; tot=(nb+1)*lh
        yt=y+h/2+tot/2-lh/2
        t1=ax.text(x+w/2,yt,title,ha="center",va="center",fontsize=FS,fontweight="bold",**SANS)
        t2=ax.text(x+w/2,yt-lh*0.62,body,ha="center",va="top",fontsize=FS,linespacing=1.3,**SANS)
        boxes.append(((t1,t2),x,y,w,h)); return (x,y,w,h)
    def harr(a,b):
        ax.annotate("",xy=(b[0],b[1]+b[3]/2),xytext=(a[0]+a[2],a[1]+a[3]/2),arrowprops=dict(arrowstyle="-|>",lw=0.7,color=EC,mutation_scale=7,shrinkA=0,shrinkB=0))
    def line(pts,arrow=True):
        xs,ys=zip(*pts); ax.plot(xs[:-1]+(xs[-1],),ys[:-1]+(ys[-1],),color=EC,lw=0.7) if not arrow else None
        if arrow:
            ax.plot(xs[:-1],ys[:-1],color=EC,lw=0.7)
            ax.annotate("",xy=pts[-1],xytext=pts[-2],arrowprops=dict(arrowstyle="-|>",lw=0.7,color=EC,mutation_scale=7,shrinkA=0,shrinkB=0))
    # phase bands
    L=0.30; bands=[(2.42,1.05,"Data"),(1.27,0.98,"Features"),(0.08,1.02,"Modelling and\nevaluation")]
    for y,h,s in bands:
        ax.add_patch(_Rect((0.02,y),L,h,fc="#e6e6e6",ec="none"))
        ax.text(0.02+L/2,y+h/2,s,rotation=90,ha="center",va="center",fontsize=6.3,fontweight="bold",**SANS)
    X0=0.42; G=0.20; W4=(FW-X0-0.03-3*G)/4; h1=0.92; y1=2.48
    b1=box(X0,y1,W4,h1,"Data","Dhaka, daily, 2023–2025\n(1,096 days, Open-Meteo)\nCAMS PM2.5, PM10, US AQI\nand weather variables")
    b2=box(X0+(W4+G),y1,W4,h1,"Quality checks","missing values, duplicates\ndate continuity\nPM2.5 ≤ PM10\nAQI reconstruction check")
    b3=box(X0+2*(W4+G),y1,W4,h1,"Feature engineering","lags k = 0, 1, 2\ntarget: next-day AQI\n1,093 samples")
    b4=box(X0+3*(W4+G),y1,W4,h1,"Chronological split","by target date\ntrain 544 | validation 184\ntest (2025) 365")
    for a,b in [(b1,b2),(b2,b3),(b3,b4)]: harr(a,b)
    y2=1.33; h2=0.86; W3=(FW-X0-0.03-2*G)/3
    c1=box(X0,y2,W3,h2,"Lag screening","pollutant-lag candidates A–D\noriginal and inner split")
    c2=box(X0+(W3+G),y2,W3,h2,"Scenario S1","AQI, PM2.5, PM10\nlags 0–2 (9 features)",fc="#f2f2f2")
    c3=box(X0+2*(W3+G),y2,W3,h2,"Scenario S2","S1 + temperature, rainfall,\nwind speed, lags 0–2\n(18 features)",fc="#f2f2f2")
    # split -> lag screening (orthogonal)
    xm=b4[0]+b4[2]/2; ymid=(y1+y2+h2)/2
    line([(xm,y1),(xm,ymid),(c1[0]+c1[2]/2,ymid),(c1[0]+c1[2]/2,y2+h2)])
    harr(c1,c2); harr(c2,c3)
    y3=0.14; h3=0.90
    d1=box(X0,y3,W3,h3,"Model training","RF, XGBoost, LightGBM\n(grid search on validation),\nlinear ARX; refit on 728 days")
    d2=box(X0+(W3+G),y3,W3,h3,"Test-year evaluation","MAE, RMSE, R², bias\nvs persistence, seasonal-naïve\nand climatology")
    d3=box(X0+2*(W3+G),y3,W3,h3,"Robustness","DM test, block bootstrap,\n10 seeds, fixed-config. ablation,\nrolling origin, permutation\nimportance")
    # S1/S2 -> model training
    ymid2=(y2+y3+h3)/2; xc2=c2[0]+c2[2]/2; xc3=c3[0]+c3[2]/2; xd1=d1[0]+d1[2]/2
    ax.plot([xc2,xc2],[y2,ymid2],color=EC,lw=0.7); ax.plot([xc3,xc3],[y2,ymid2],color=EC,lw=0.7)
    line([(xc3,ymid2),(xd1,ymid2),(xd1,y3+h3)])
    harr(d1,d2); harr(d2,d3)
    # fit check: every title and body must sit inside its box with a margin
    fig.canvas.draw(); r=fig.canvas.get_renderer(); inv=ax.transData.inverted()
    for (t1,t2),x,y,w,h in boxes:
        for t in (t1,t2):
            bb=t.get_window_extent(r).transformed(inv)
            assert bb.x0>=x+0.05 and bb.x1<=x+w-0.05 and bb.y0>=y+0.04 and bb.y1<=y+h-0.04, "overflow: "+t.get_text()[:30]
    print("fig1: all text inside boxes")
    fig.savefig(path,dpi=300,bbox_inches=None,pad_inches=0); _plt.close(fig)
make_fig1()


In [ ]:
# ---- Fig 2 time series ----
vars_=[("AQI_Level","AQI"),("PM2.5","PM2.5\n(µg/m³)"),("PM10","PM10\n(µg/m³)"),("Avg_Temperature_C","Temp.\n(°C)"),("Rainfall_mm","Rain\n(mm)"),("Max_Wind_Speed_kmh","Wind\n(km/h)")]
fig,axs=plt.subplots(7,1,figsize=(W,5.4),sharex=True)
for i,(c,l) in enumerate(vars_):
    a=axs[i]; a.plot(df.index,df[c],color=C1 if i<3 else C2,lw=0.6); a.set_ylabel(l); lab(a,f"({chr(97+i)})",0.035,0.70)
r=df["PM2.5"]/df["PM10"]; axs[6].plot(df.index,r,color=DK,lw=0.6); axs[6].set_ylabel("PM2.5/\nPM10"); lab(axs[6],"(g)",0.035,0.70)
for a in axs:
    a.axvspan(pd.Timestamp("2024-07-01"),pd.Timestamp("2024-12-31"),color="#000",alpha=0.06,lw=0); a.axvspan(pd.Timestamp("2025-01-01"),pd.Timestamp("2025-12-31"),color=C1,alpha=0.06,lw=0)
    a.axvline(pd.Timestamp("2024-11-12"),color=DK,ls="--",lw=0.6); a.grid(axis="y",lw=0.3,alpha=0.4)
axs[0].text(pd.Timestamp("2023-09-01"),225,"training",fontsize=6.5); axs[0].text(pd.Timestamp("2024-08-01"),225,"validation",fontsize=6.5); axs[0].text(pd.Timestamp("2025-05-15"),225,"test",fontsize=6.5)
axs[6].text(pd.Timestamp("2024-11-05"),0.90,"CAMS cycle 49R1 upgrade (12 Nov 2024)",fontsize=6.5,ha="right")
axs[-1].set_xlim(pd.Timestamp("2023-01-01"),pd.Timestamp("2025-12-31")); axs[-1].xaxis.set_major_locator(matplotlib.dates.MonthLocator(bymonth=[1,5,9]))
axs[-1].xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %Y")); axs[-1].set_xlabel("Date (data: 1 January 2023–31 December 2025)")
fig.align_ylabels(axs); fig.savefig("figures/fig2_timeseries.png"); plt.show()

In [ ]:
# ---- Fig 3 test metrics ----
T=R["test"]
if A.original_main:
    T=dict(T); orig={"RF-S1":(9.8815,13.0543,0.9150),"RF-S2":(9.7128,12.7453,0.9190),"XGB-S1":(9.8453,12.9490,0.9164),"XGB-S2":(8.8454,11.9138,0.9292),"LGB-S1":(10.0344,13.1980,0.9131),"LGB-S2":(9.4648,12.4665,0.9225)}
    for k,(m,r_,q) in orig.items(): T[k]=dict(MAE=m,RMSE=r_,R2=q)
names=["RF","XGB","LGB"]; disp={"RF":"Random Forest","XGB":"XGBoost","LGB":"LightGBM"}
fig,axs=plt.subplots(1,3,figsize=(W,2.0))
for j,(met,yl,ref) in enumerate([("RMSE","RMSE (AQI)",T["Persistence--"]["RMSE"]),("MAE","MAE (AQI)",T["Persistence--"]["MAE"]),("R2","R²",T["Persistence--"]["R2"])]):
    a=axs[j]; x=np.arange(3); w=0.36
    v1=[T[f"{n}-S1"][met] for n in names]; v2=[T[f"{n}-S2"][met] for n in names]
    a.bar(x-w/2-0.01,v1,w,color=C1,label="S1",zorder=3); a.bar(x+w/2+0.01,v2,w,color=C2,label="S2",zorder=3)
    a.axhline(ref,color=DK,ls="--",lw=0.7,zorder=4); a.text(-0.55,ref,"persistence",fontsize=6,va="bottom",ha="left")
    a.set_xticks(x); a.set_xticklabels(["RF","XGBoost","LightGBM"]); a.set_ylabel(yl); a.grid(axis="y",lw=0.3,alpha=0.4,zorder=0)
    if met=="R2": a.set_ylim(0.80,0.95)
    else: a.set_ylim(0,max(max(v1),ref)*1.18)
    lab(a,f"({chr(97+j)})",-0.12)
axs[0].legend(frameon=False,loc="upper right",ncol=2,bbox_to_anchor=(1.0,1.12)); fig.tight_layout(w_pad=1.2); fig.savefig("figures/fig3_metrics.png"); plt.show()

In [ ]:
# ---- Fig 4 grouped permutation importance ----
G=["PM25","PM10","AQI","Rain","Wind","Temp"]; GL={"PM25":"PM2.5","PM10":"PM10","AQI":"AQI","Rain":"Rainfall","Wind":"Wind speed","Temp":"Temperature"}
fig,axs=plt.subplots(1,3,figsize=(W,2.1),sharey=True)
xmax=max(max(R["gpi"][a][p][g][0]+R["gpi"][a][p][g][1] for g in G for p in ("val","test")) for a in names)*1.08
for j,a in enumerate(names):
    ax=axs[j]; y=np.arange(len(G)); h=0.38
    for k,(p,c,l) in enumerate([("val",GR,"2024 validation"),("test",C2,"2025 test")]):
        m=[R["gpi"][a][p][g][0] for g in G]; s=[R["gpi"][a][p][g][1] for g in G]
        ax.barh(y+(k-0.5)*h,m,h*0.95,xerr=s,color=c,label=l,error_kw=dict(lw=0.6,capsize=1.5),zorder=3)
    ax.set_xscale("symlog",linthresh=1); ax.set_xlim(-0.5,xmax); ax.axvline(0,color=DK,lw=0.5)
    ax.set_yticks(y); ax.set_yticklabels([GL[g] for g in G]); ax.invert_yaxis(); ax.set_xlabel("ΔRMSE after group permutation"); ax.set_title(disp[a]); lab(ax,f"({chr(97+j)})",-0.05)
    ax.grid(axis="x",lw=0.3,alpha=0.4,zorder=0)
axs[0].legend(frameon=False,loc="lower right"); fig.tight_layout(w_pad=0.8); fig.savefig("figures/fig4_gpi.png"); plt.show()

In [ ]:
# ---- Fig 5 actual vs predicted ----
fig,axs=plt.subplots(3,1,figsize=(W,4.6),sharex=True)
for j,a in enumerate(names):
    ax=axs[j]; ax.plot(P.index,P["y"],color=DK,lw=0.8,label="Actual"); ax.plot(P.index,P[f"{a}-S1"],color=C1,lw=0.6,label="S1"); ax.plot(P.index,P[f"{a}-S2"],color=C2,lw=0.6,label="S2")
    ax.set_ylabel("AQI"); ax.set_title(disp[a],loc="left",x=0.01,y=0.86); lab(ax,f"({chr(97+j)})",-0.06,0.8); ax.grid(axis="y",lw=0.3,alpha=0.4)
    ax.set_ylim(20,330); ins=inset_axes(ax,width="22%",height="42%",loc="upper center",bbox_to_anchor=(0.12,0,1,1),bbox_transform=ax.transAxes,borderpad=0.3)
    z=P.loc["2025-01-18":"2025-01-31"]; ins.plot(z.index,z["y"],color=DK,lw=0.9,marker="o",ms=1.8); ins.plot(z.index,z[f"{a}-S1"],color=C1,lw=0.8); ins.plot(z.index,z[f"{a}-S2"],color=C2,lw=0.8)
    ins.tick_params(labelsize=5.5); ins.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%d")); ins.set_xticks(pd.date_range("2025-01-19","2025-01-31",freq="4D"))
    ins.axvspan(pd.Timestamp("2025-01-22"),pd.Timestamp("2025-01-25"),color=C2,alpha=0.12,lw=0); ins.set_title("18–31 Jan 2025",fontsize=5.5,pad=1)
axs[0].legend(frameon=False,loc="upper left",bbox_to_anchor=(0.17,1.02),ncol=3,handlelength=1.2,columnspacing=0.8)
axs[-1].set_xlim(pd.Timestamp("2025-01-01"),pd.Timestamp("2025-12-31")); axs[-1].xaxis.set_major_locator(matplotlib.dates.MonthLocator(bymonth=[1,3,5,7,9,11]))
axs[-1].xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b %Y")); axs[-1].set_xlabel("Target date (2025)")
fig.savefig("figures/fig5_predictions.png"); plt.show()

In [ ]:
# ---- Fig 6 residuals XGB-S2 ----
res=P["y"]-P["XGB-S2"]
fig,axs=plt.subplots(1,2,figsize=(W,1.9),gridspec_kw=dict(width_ratios=[1,2.2]))
axs[0].hist(res,bins=30,color=C2,edgecolor="white",lw=0.4); axs[0].axvline(0,color=DK,lw=0.6); axs[0].axvline(res.mean(),color=C1,ls="--",lw=0.8)
axs[0].set_xlabel("Residual (actual − predicted)"); axs[0].set_ylabel("Days"); lab(axs[0],"(a)",-0.15)
axs[1].plot(res.index,res,color=C2,lw=0.6); axs[1].axhline(0,color=DK,lw=0.6); axs[1].set_ylabel("Residual"); lab(axs[1],"(b)",-0.06)
axs[1].set_xlim(pd.Timestamp("2025-01-01"),pd.Timestamp("2025-12-31")); axs[1].xaxis.set_major_locator(matplotlib.dates.MonthLocator(bymonth=[1,3,5,7,9,11]))
axs[1].xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b")); axs[1].grid(axis="y",lw=0.3,alpha=0.4)
fig.tight_layout(); fig.savefig("figures/fig6_residuals.png"); plt.show()

In [ ]:
# ---- Fig 7 AQI reconstruction ----
def sub(c,bp):
    for Cl,Ch,Il,Ih in bp:
        if c<=Ch: return (Ih-Il)/(Ch-Cl)*(c-Cl)+Il
    Cl,Ch,Il,Ih=bp[-1]; return (Ih-Il)/(Ch-Cl)*(c-Cl)+Il
pm25=[(0,12,0,50),(12.1,35.4,51,100),(35.5,55.4,101,150),(55.5,150.4,151,200),(150.5,250.4,201,300),(250.5,350.4,301,400),(350.5,500.4,401,500)]
pm10=[(0,54,0,50),(55,154,51,100),(155,254,101,150),(255,354,151,200),(355,424,201,300),(425,604,301,500)]
c25=(0.5*(df["PM2.5"]+df["PM2.5"].shift(1))); c10=(0.5*(df["PM10"]+df["PM10"].shift(1)))
rec=np.maximum(c25.apply(lambda c:sub(c,pm25) if c==c else np.nan),c10.apply(lambda c:sub(c,pm10) if c==c else np.nan))
d0=df["PM2.5"].apply(lambda c:sub(c,pm25))
m=rec.notna(); pre=df.index<pd.Timestamp("2024-11-12")
fig,axs=plt.subplots(1,2,figsize=(W,2.3))
for k,(xx,t) in enumerate([(d0,"same-day mean PM2.5"),(rec,"two-day mean PM")]):
    a=axs[k]; mm=xx.notna()
    a.scatter(xx[mm&pre],df["AQI_Level"][mm&pre],s=3,color=C1,alpha=0.6,lw=0,label="before 12 Nov 2024")
    a.scatter(xx[mm&~pre],df["AQI_Level"][mm&~pre],s=3,color=C2,alpha=0.6,lw=0,label="from 12 Nov 2024")
    a.plot([0,260],[0,260],color=DK,lw=0.6,ls="--"); a.set_xlim(0,260); a.set_ylim(0,260)
    rr=np.corrcoef(xx[mm],df["AQI_Level"][mm])[0,1]; mae=np.mean(np.abs(xx[mm]-df["AQI_Level"][mm]))
    a.text(8,240,f"r = {rr:.3f}; MAE = {mae:.2f}",fontsize=7); a.set_xlabel(f"Reconstructed AQI ({t})"); a.set_ylabel("Reported daily AQI"); lab(a,f"({chr(97+k)})",-0.12)
axs[0].legend(frameon=False,loc="lower right",markerscale=3); fig.tight_layout(); fig.savefig("figures/fig7_aqi_recon.png"); plt.show()
print("All figures saved in the figures/ folder")

## 16. Download the results (optional, Colab)

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("hasil_revisi_aqi", "zip", ".", None)
    for f in ["results.json", "test_predictions.csv"]: files.download(f)
except ImportError:
    print("Not Colab: files are already saved in the working folder.")